## CASH Notebook

## Celestial Chase - LVL 1: The Teal Beacon

You've just woken up from cryo-sleep. No memory. No crew. Just you, a dying sun, and a faint signal pulsing from the sky.

One star glows different from the rest. Not white. Not grey. **Teal.**

Find it. Its position holds part of the key. But position alone is not enough - you must also know how much of its face is lit by the sun.

---

**The encoded signal:** `ejytupr`

**Your task:**
1. Display the image and find the **cyan pixel** - it is the only pixel where `B == 255` and `G == 255` and `R == 0`
2. Read its `(x, y)` coordinates
3. Use `ephem` to compute **Venus's phase** (`int(venus.phase)`) on `2025/6/21 12:00:00` UTC from Zurich (lat=`47.3769`, lon=`8.5417`)
4. Build the key: `key = x * y + int(venus.phase)`
5. Build the permutation: `random.Random(key).shuffle(list(range(len(encoded))))`
6. Reverse the transposition to decode: `decoded[i] = encoded[perm[i]]`


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

# Starfield image embedded directly in this notebook
img_b64 = "iVBORw0KGgoAAAANSUhEUgAAAZAAAAGQCAIAAAAP3aGbAAASHklEQVR4Ae3Ba3CWhZnH4f/9cWe7tSIQiyNFPBcFzdCKBVFrVRQECSScIxEEApEACeEYCIRjSIBgIIBgMJwTCIKgUWtVhIotEwWlnpHiSBtArN3u7MfsDDvO1NFITu/7Pvf7/K7LhGbYt2dP3/79BSAqTABayL+++eYnl10mNM+xmpouiYn6ISYAcMIEAE6YAMAJEyJs186dAwcNEoBmM0n5eXm5eXlCnMrMyCguKRHgnwkAnDABgBMmAHDChOjKzMgoLikRgMYzAZG3u6JiQEqKgOYxwZsTx4936txZaKRHHn74+RdeEDwzAWGyZ9eu/gMHCj6ZAM/++x//+K+f/Uzw5vNPP73muuvUSCbgUt4/duyWLl0ExJoJAJwwAYATJgBwwgQgpl7cv/+hPn2EBjChYYYNHrx1xw6h2X7/0ku/e/BBIWTKy8pS09LUPCbArR1btw4eNkwIDRMAOGEC0DCZGRnFJSVC7Jia5MvTp69q316xsHrVqgkTJwpA+JgAwAkTQmZwcvKOykrBv2M1NV0SExUmJji3sqhoUlaWGmb2jBkLFi8WfDr5yScdr79eIWYCACdMAOCECQCcMMWdYzU1XRITBSDumABE3qaNG0eOGiU0jwkAnDABgBOmBps3Z87c+fMFhNKi/PyZubmC9OGJEzd16qRYMLl1vra2dUKCAISGCQifooKCrJwcwRsTAFeWLFw4fdYshZIJiIX/rKv7HzNF2JhRo9Zv3CjECxNw0epVqyZMnKig2ltV1S8pSRd9ff785a1by5shKSnbKyqEZjABgBMmAHDCBABOmADACROA+h187bWe994rXJSRnl5SWqrYMQGAEyYA+FFby8uHpaYqAEy4lNFpaRvKyoRvJfXrV7V3r4CoMwGAEyYAcMIEAE6YAMAJE4CLBicn76isVIi9c/To7V27KsBMiIUP3n//5ltuUbDlZGUVFBUJ31q9atWEiRP1Q1KHDSvfulWIMBMAOGECEGAlxcUZmZnCRSbEr/O1ta0TEoRw2FpePiw1VXHNBABOmADACRMAOGECguTwwYPde/ZUJD05fvxTa9YIDpki7FhNTZfERCF8FuXnz8zNFdByTECIjRo5cuOmTYITpkj69KOPrrvxRqFh8nJz8/LzBaAeJgBwwoQAW1lUNCkrSwAuMqHlVGzfnjJkiIDGKC8rS01LExrAhGg5X1vbOiFBAJrKhHh3YN++3n37CvDPBDTY2b/9re3Pf64WlTVpUtHKlQIawNRUH3/wwQ033ywg9HJnzsxftEiIPBPQAB9/8MENN98sePDnI0d+1a2b4pEJLe2rs2evaNtWAFqaCYB/B197ree99yremQAg2JL69avau1eSCQCcMAGAEyagSQ698UaPu+8WEEUmxJHndu9+dMAAAXHKhGiZOGHCqtWrBaCpTAB82lpePiw1VWFiAhpg3BNPrH36aQExZQIAJ0wA4IQJ4ba7omJASooAD0wA4IQJQOwceuONHnffLTSMKb78/qWXfvfggwKiZdPGjSNHjRKiwuTQvj17+vbvLyA0pmRmLi8uVuiZgm10WtqGsjKhefr27r3vwAEBzpkAwAkTADhhAgAnTADghCnWjr79dtc77hCAYNhaXj4sNVWBZAIAJ0wA4IQJAKJl86ZNI0aOVFOZEDHjx45ds26d/Dhx/Hinzp0FBJUJkdG7V68D1dUC0HJMAOCECQCcMAGAEyYAcMIEIABOHD/eqXNnBcA/v/76p5dfrkAyNUlpSUl6RoYAIIpMANxat2bN2PHjFRomAFH31MqVT06aJDSSqR7/UVf3v2YC4kvburqzZoJPJgAO5WRlFRQVKWRMCI0vT5++qn17AW6ZAMAJEwA4YQIAJ0xBcmDfvt59+wqIheFDhmzZvl0IMBMAOGECACdMaJiN69ePGjNGAGLHBEhfnDp1dYcOAoLNBABOmAAgdk599lmHa69Vw5gAwAkTgOarq5OZEGEmAM1UV6f/ZyYPztfWtk5IkEMmV46/807n228XEDR1dTJTy3nr0KE7e/QQvssEePbPr7/+6eWXC+FgAgAnTADghMmJGTk5iwsKBCDETADghCk0Bvbvv2vPHgEBkD5mTOn69UIjmQDACROAptq3Z0/f/v2FaDEBAXbh3LlWbdoIuMgEAM3wyYcfXn/TTYoKU4hlTZpUtHKlADhhAgAnTHDlzddfv+ueewSEkgkAnDABzZM6bFj51q0CIs9Uv22bNw8dMUJ+fPrRR9fdeKMgbdu8eeiIEWqw4UOGbNm+XUCwmQDACRPQJO8cPXp7164CosgEAE6YANRjWnb20sJCITBMFxUuXZo9bZqA4BmdlrahrEyAZAqqE8ePd+rcWUAz9Hnoof0vvqgIWF9aOiY9XYguU4D9+ciRX3XrJgC4yAQATpgAwAlTrM2fO3fOvHkCgEsxAYATJgBwwhRg69asGTt+vIJt+tSpS5YtE4DIM6Exfv/SS7978EEBiAUTADhhAgAnTADghAkAnDABgBMmAHDCBCDWBicn76isVGRs3rRpxMiRigumFvLWoUN39uiheLFh3brRY8cKQJCYgGA7cvhwt+7dBUgmAHDCBABOmADACRPg0IcnTtzUqZMu5b133731ttuEeGECEBpZkyYVrVwpt0wAGiypX7+qvXuFGDEBgBMmAHDCBITPH1555bf33y94YwK+9deTJ3/RsaOAgJk1ffrCJUskmYBoqdyxI3nwYMWFP7311q/vvFOILhMAOGGCE9OnTl2ybJmAEDMF1cHXXut5771y64tTp67u0EEAWo4JAJwwAYATJgAxlT15cuGKFWohSxYunD5rluKUCQCcMAENNjg5eUdlpYAYMQGAEyb4MTg5eUdlpYCwMiHAhg8ZsmX7dgG4yBQa5WVlqWlpAuCWCQCcMAGIF9OnTl2ybJnqUX3gQK/evdVso9PSNpSVKRZMAOCECfieooKCrJwcAQFjQrT8/csvr7zqKgFoKhMAOGECcClfnT17Rdu2QqyZAMAJE9Byli9bNmXqVAEN9pf33vvlrbeqYUwA4IQJAVOweHHOjBkKn6Nvv931jjsUYLkzZ+YvWqSL3jp06M4ePYToMgGAEyYgAKoqK5OSkxUm69asGTt+vJpqRk7O4oICNc+KwsLJ2dnywwTErzGjRq3fuFGIFyYAcMIEAE6YAMAJE+BWWmpqWXm56vH8c8898uijQhwxIcSSk5Iqq6oEJ/r16bN3/36FmAkAnDABgBOmyHiluvr+Xr3USMffeafz7bcLAH6ICQiZ3RUVA1JSFDKDBg7cuWuXnDMBgBMmAHDC1KI2rFs3euxYeVO5Y0fy4MGSXty//6E+fQQgkEwA4IQJAJwwOXTk8OFu3bsLQMiY4t2hN97ocffdiqJnn3nmsccfl1tVlZVJycmqR/Hy5ZlTpgiIBRMAOGECACdMgBOvvvzyfQ88IISYCc124dy5Vm3aCECEmVpCUUFBVk6OAITJ0EGDtu3cqSgyAYATJuC7as+cSWjXTkAzrF29etyECWpppniXl5ubl58vAIHx3O7djw4YoMYzAc1Te+ZMQrt2AiLPBETAqy+/fN8DDwhoUSY4t3PbtkFDhwoIjK/Onr2ibVtFgAkIhqxJk4pWrlTj/eubb35y2WVCS/jryZO/6NhRQWUCACdMwTZ39ux5CxYIACQTcFFpSUl6RoaAADMB8GPBvHmz585VWJkAwAkTADhhAoAmqT5woFfv3ooiEwD8qEX5+TNzcxUAJgBuvVJdfX+vXgoNEwA4Yfo3fz158hcdOwqAK59/+uk1112nEDABgBMmAHDCBMS111999Z777hPcWrxgwYzZs3WRCT6VlpSkZ2QICBMTgPrt2bWr/8CBQjCYAMAJUwNU7tiRPHiwwm3BvHmz584VYqdg8eKcGTOEEDM51+ehh/a/+KIAhIAJAJwwBcz52trWCQkCECZfnDp1dYcOuhQTAD8+/eij6268UWFlAprkL++998tbbxUio6S4OCMzU/guE4DvmThhwqrVq4WAMQFA5PXu1etAdbWaxwQADbO7omJASopixxQmhUuXZk+bJgA+mQDACRMAOGGKsPePHbulSxcBQLOZAMAJUwMsys+fmZsrhNjmTZtGjBwpIKZMAOCECQCcMAGAE6b4sm3z5qEjRghAPDIBgBMmoDFKioszMjMFXMqcWbPmL1yoFmUCACdMCJlHHn74+RdeEOCQCUDzHKup6ZKYKESeqXm+Pn/+8tatBQCRZ4J/5WVlqWlpCqRjNTVdEhMFtAQTADhhCqTDBw9279lTAPBvTADghAn1WLpo0bSZMwUgMEz4rp/X1f3NTN81e8aMBYsXC5E3a/r0hUuWCPghJjTJ66++es999wkIk88+/vjaG25Q7JgAwAkTADhhAgAnTIg7VZWVScnJCqttmzcPHTFCiEcmAHDCBABOmC7aW1XVLylJwfDl6dNXtW8vAPguEwA4YYqi/Xv39unXTwDQJCZE2Ksvv3zfAw8I+NbuiooBKSlC45kAwAkTADRefl5ebl6eossUVOtLS8ekpwsenP788/bXXCMgwkwA4IQJIXDyk086Xn+9AOdMQHSd+/vf21x5pYDGMwGAEyYAwdC7V68D1dVC/UwA4IQJQHQdPniwe8+eQuOZYuTCuXOt2rRRC1m7evW4CRMEIK6ZAMAJE4B6DOzff9eePUJgmKLlq7Nnr2jbVgDqd/rzz9tfc41QDxMAOGEC4lHqsGHlW7cKLWfntm2Dhg6VVHvmTEK7dooFU/DMnzt3zrx5wg8pXr48c8oUAaFkAgAnTADghMm/DevWjR47VkC8OFZT0yUxUfgeEwA4YfohC+fPnzVnjoDGyMzIKC4pERAxpmhZWVQ0KStLANBUpu/auW3boKFDhQBbOH/+rDlzBISPCQCcMAGAEyYAcMIE1ON8bW3rhARFRb8+ffbu3y/gR5kiZuzo0es2bBAAtBATADhhAgAnTIAH+/fu7dOvnxBupm+tLy0dk54uxLulixZNmzlT4TZ7xowFixcrTAYNHLhz1y45Zwql9aWlY9LTBdTjvXffvfW224SAMQGAEybEoyEpKdsrKgQnXqmuvr9XL+FSTADghAkAnDAhWl54/vmHH3lEAJrKBDTYs88889jjjwuIERMQSY8NH/7sli0CWoIJAJwwwYMjhw93695dQLiZEAJ7q6r6JSUJcM4EAE6YAMAJE8Lh+eeee+TRRwV4ZgIAJ0wAEGH5eXm5eXlqNhMQUxXbt6cMGSKgAUwAgiR78uTCFSsUGjlZWQVFRWoYkxNt6urOmQke5OXm5uXnC2hpJgBwwoRYuHDuXKs2bQSgMUwA4IQp3P701lu/vvNOAfDAhIjZsG7d6LFjhbi2b8+evv37C9KIoUM3b9umSDIBgBMmAHDC1Eh/fPPN39x1l8LnH1999bMrrhCA2DEBgBMmAHDCBABOmNyaOmXKsuXLBSA0TI20orBwcna2gHB7bvfuRwcMEKLLBP/e+MMf7v7tbwXEOxPgypZnnx3+2GNCKJkAwAkTADhhQktYOH/+rDlzBCCSTH6kjxlTun69AISVCQCcMAGAEyYAiJbJEyeuWLVKTWVC7Lz5+ut33XOPADSMqTGeXrv2iXHjhMCY9OSTK596SkA4mELjmwsXLmvVSmiq7Vu2DBk+XMD3zJszZ+78+Yo8EwA4YYIfi/LzZ+bmCggrEwA4YQIAJ0wA4IQJHixesGDG7NkCws2EyPvjm2/+5q67FA5HDh/u1r27gAgw4VIGJyfvqKwUgFgzAQiTzz7++NobbpBPphY1YujQzdu2yaGS4uKMzEyh5UwYN2712rVCvCjbsCFt9GjFlAkAnDABgBMmAHDChJa2orBwcna2ALQ0EwA4YQIAJ0yBUXvmTEK7dkKzzZw2bdHSpQLijgkAnDAFQ6u6ugtmAoD6mRA7yUlJlVVVAtAwJsSpU5991uHaaxV4x2pquiQmKtjOfPFFu6uvFmLNFDG7KyoGpKQIQFwoXLo0e9o0xZTJgyEpKdsrKgTgRz3+2GPPPPus4pcJCJjCpUuzp00T8D0mAHDCBACBl5GeXlJaagIAJ0yIrvlz586ZN0+N8fTatU+MGycPxowatX7jRqGFtKqru2AmfMsEAE6YAMAJEwA4YQIAJ0zNUH3gQK/evQUAUWH6USsKCydnZyuo3jl69PauXQUgruVkZRUUFUn6PxcQP83h7Sa4AAAAAElFTkSuQmCC"

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('starfield.png', img)
display(IPImage('starfield.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, random

encoded  = "ejytupr"
obs_date = "2025/6/21 12:00:00"
obs_lat  = "47.3769"
obs_lon  = "8.5417"

# TODO Step 1: Find the cyan pixel (B=255, G=255, R=0) in img
# Hint: use np.where or a loop
pixel_x, pixel_y = 0, 0  # replace with actual coordinates

# TODO Step 2: Compute Venus phase with ephem
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date
venus = ephem.Venus()
venus.compute(obs)
phase = 0  # replace: int(venus.phase)

# TODO Step 3: Build key and permutation
key  = pixel_x * pixel_y + phase
perm = list(range(len(encoded)))
random.Random(key).shuffle(perm)

# TODO Step 4: Reverse the transposition
# Hint: if encoded[perm[i]] = original[i], then decoded[i] = encoded[perm[i]]? Think carefully.
def transpose_decode(encoded, perm):
    pass  # implement this

answer = transpose_decode(encoded, perm)
print(answer)
